In [1]:
import pandas as pd
import plotly.express as px
import squarify
import numpy as np

# =============================
# Dataset Loading
# =============================

games_df = pd.read_csv("../Data/api data/Old data/Final Database/games.csv")
detailed_games_df = pd.read_csv("../Data/api data/Old data/Final Database/detailed_games.csv")
company_games_df = pd.read_csv("../Data/api data/Old data/Final Database/company_games.csv")
genre_games_df = pd.read_csv("../Data/api data/Old data/Final Database/genre_games.csv")
platform_df = pd.read_csv("../Data/api data/Old data/Final Database/platform.csv")


final_df = pd.read_csv("../Data/Dataset.csv")

C:\Users\laure\AppData\Local\Temp\ipykernel_5432\1055050788.py:14: DtypeWarning: Columns (8,25,33) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv("../Data/Dataset.csv")


In [2]:


# Flatten all genre lists and extract unique genres
unique_genres = set(
    g.strip()
    for genre_str in final_df["genres"].dropna()
    if isinstance(genre_str, str)
    for g in genre_str.split(",")
)


# Convert to a sorted list if needed
unique_genres = sorted(unique_genres)

# Print or return them
print(unique_genres)



['Action', 'Adventure', 'Arcade', 'Board Games', 'Card', 'Casual', 'Educational', 'Family', 'Fighting', 'Indie', 'Massively Multiplayer', 'Platformer', 'Puzzle', 'RPG', 'Racing', 'Shooter', 'Simulation', 'Sports', 'Strategy']


In [40]:
import dash
from dash import dcc, html, Input, Output, State, ctx, ALL
import dash_bootstrap_components as dbc
import threading
import webbrowser
import plotly.express as px
import plotly.graph_objects as go
import squarify

# Create Dash app with Bootstrap theme
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
app.title = "Dashboard"

years = list(range(1970, 2025))

# =========================
# Helper component builders
# =========================

def build_top_control(active_tab):
    return dbc.ButtonGroup([
        dbc.Button("Games", id="games-button", n_clicks=0, color="primary" if active_tab == "games" else "secondary"),
        dbc.Button("Trends", id="trends-button", n_clicks=0, color="primary" if active_tab == "trends" else "secondary")
    ], id="top_control", style={"width": "100%"})

def build_search_bar():
    return dbc.Card(
        dbc.CardBody([
            dbc.Input(
                id="search_bar",
                placeholder="Search...",
                type="text",
                style={"marginBottom": "0"}
            )
        ]),
        style={
            "marginTop": "1rem",
            "marginBottom": "8rem",
            "boxShadow": "0px 2px 6px rgba(0,0,0,0.1)",
            "border": "1px solid #ced4da",
            "borderRadius": "0.5rem",
            "backgroundColor": "white"
        }
    )


def build_data_view():
    return html.Div(
        children=[
            html.Div(
                html.Div(id="main_graph", style={"width": "100%"}),
                #dcc.Graph(id="main_graph", style={"height": "600px", "width": "100%"}),
                style={
                    "overflowX": "auto",
                    "width": "100%",
                    "paddingBottom": "1rem"
                }
            )
        ],
        id="data_view",
        style={"padding": "2rem"}
    )



def build_games_middle(selected_sub, selected_sort_options, selected_genres):
    options = ["Most Popular", "Genres"]
    buttons = [
        dbc.Button(
            label,
            id={"type": "sub-button", "index": label.lower().replace(" ", "-") + "-sub"},
            color="primary" if selected_sub == label.lower().replace(" ", "-") + "-sub" else "secondary",
            n_clicks=0,
            style={"width": "100%", "marginBottom": "0.5rem"}
        )
        for label in options
    ]

    if selected_sub == "most-popular-sub":
        sort_options = ["Added", "Rating", "YouTube", "Twitch", "Metacritic"]
    elif selected_sub == "genres-sub":
        sort_options = ['Action', 'Adventure', 'Arcade', 'Board Games', 'Card', 
         'Casual', 'Educational', 'Family', 'Fighting', 'Indie',
           'Massively Multiplayer', 'Platformer', 'Puzzle',
             'RPG', 'Racing', 'Shooter', 'Simulation',
               'Sports', 'Strategy']

    sort_buttons = [
        dbc.Checkbox(
            label=m,
            id={"type": "sort-button", "index": m},
            value=m in selected_sort_options,
            style={"marginBottom": "0.4rem"}
        )
        for m in sort_options
    ]


    return html.Div([
        dbc.Row([
            dbc.Col(buttons, width=6),
            dbc.Col([
                html.H6("Sort By"),
                *sort_buttons
            ], width=6)
        ]),

        # ── Release-year slider (unchanged) ────────────────────────────────
        html.Div([
            html.Label("Select Release Year Range", style={"fontWeight": "bold", "marginTop": "1rem"}),
            dcc.RangeSlider(
                id="year-range-slider",
                min=1970,
                max=2024,
                step=1,
                marks={y: str(y) for y in range(1970, 2025, 10)},
                value=[2000, 2020],
                tooltip={"placement": "bottom", "always_visible": False},
                allowCross=False,
                updatemode="mouseup"
            )
        ], style={
            "marginTop": "1.5rem", "padding": "1rem",
            "backgroundColor": "#f1f3f5", "borderRadius": "0.5rem",
            "boxShadow": "inset 0 1px 3px rgba(0,0,0,0.1)", "width": "100%"
        }),

        # ── NEW: number-of-games slider ────────────────────────────────────
        html.Div([
            html.Label("Number of games (3 – 200)", style={"fontWeight": "bold", "marginTop": "1rem"}),
            dcc.Slider(
                id="num-games-slider",
                min=3, max=200, step=1, value=50,                 # default 50
                marks={i: str(i) for i in range(10, 201, 30)},
                updatemode="drag", tooltip={"placement": "bottom", "always_visible": False}
            )
        ], style={
            "marginTop": "1.5rem", "padding": "1rem",
            "backgroundColor": "#f1f3f5", "borderRadius": "0.5rem",
            "boxShadow": "inset 0 1px 3px rgba(0,0,0,0.1)", "width": "100%"
        })
    ])

def build_sidebar(active_tab, selected_sub, selected_sort_options, selected_genres):
    # === Games tab controls ===

    games_buttons = html.Div(
        build_games_middle(selected_sub, selected_sort_options, selected_genres),
        style={"display": "block" if active_tab == "games" else "none"}
    )

    games_year_slider = html.Div([
        html.Label("Select Release Year Range", style={"fontWeight": "bold", "marginTop": "1rem"}),
        dcc.RangeSlider(
            id="games-year-range-slider",
            min=1970,
            max=2024,
            step=1,
            marks={y: str(y) for y in range(1970, 2025, 10)},
            value=[2000, 2020],
            tooltip={"placement": "bottom", "always_visible": False},
            allowCross=False,
            updatemode="mouseup"
        )
    ], style={
        "marginTop": "1.5rem", "padding": "1rem",
        "backgroundColor": "#f1f3f5", "borderRadius": "0.5rem",
        "boxShadow": "inset 0 1px 3px rgba(0,0,0,0.1)", "width": "100%",
        "display": "block" if active_tab == "games" else "none"
    })

    games_num_slider = html.Div([
        html.Label("Number of games (3 – 200)", style={"fontWeight": "bold", "marginTop": "1rem"}),
        dcc.Slider(
            id="games-num-games-slider",
            min=3, max=200, step=1, value=50,
            marks={i: str(i) for i in range(10, 201, 30)},
            updatemode="drag"
        )
    ], style={
        "marginTop": "1.5rem", "padding": "1rem",
        "backgroundColor": "#f1f3f5", "borderRadius": "0.5rem",
        "boxShadow": "inset 0 1px 3px rgba(0,0,0,0.1)", "width": "100%",
        "display": "block" if active_tab == "games" else "none"
    })

    # === Trends tab controls ===
    trends_controls = html.Div([
        html.Label("Show games by", style={
            "fontWeight": "bold", "fontSize": "1.1rem", "marginBottom": "0.5rem"
        }),

        html.Div([
            dbc.Checkbox(
                id={"type": "trends-metric", "index": "rating"},
                label="Rating",
                value=True,
                style={
                    "display": "block",
                    "marginBottom": "0.6rem",
                    "fontSize": "1rem",
                    "marginLeft": "0.25rem"
                }
            ),
            dbc.Checkbox(
                id={"type": "trends-metric", "index": "added"},
                label="Added",
                value=False,
                style={
                    "display": "block",
                    "marginBottom": "0.6rem",
                    "fontSize": "1rem",
                    "marginLeft": "0.25rem"
                }
            ),
            dbc.Checkbox(
                id={"type": "trends-metric", "index": "metacritic"},
                label="Metacritic",
                value=False,
                style={
                    "display": "block",
                    "marginBottom": "0.6rem",
                    "fontSize": "1rem",
                    "marginLeft": "0.25rem"
                }
            ),
            dbc.Checkbox(
                id={"type": "trends-metric", "index": "youtube_count"},
                label="YouTube",
                value=False,
                style={
                    "display": "block",
                    "marginBottom": "0.6rem",
                    "fontSize": "1rem",
                    "marginLeft": "0.25rem"
                }
            ),
            dbc.Checkbox(
                id={"type": "trends-metric", "index": "twitch_count"},
                label="Twitch",
                value=False,
                style={
                    "display": "block",
                    "marginBottom": "1rem",
                    "fontSize": "1rem",
                    "marginLeft": "0.25rem"
                }
            )
        ]),

        html.Label("Other Options", style={
            "fontWeight": "bold", "fontSize": "1.1rem", "marginTop": "1rem"
        }),

        dbc.Checkbox(
            id="show-trendline-checkbox",
            value=False,
            label="Show trendline",
            style={
                "display": "block",
                "marginBottom": "0.6rem",
                "fontSize": "1rem",
                "marginLeft": "0.25rem"
            }
        ),

        dbc.Checkbox(
            id="show-genre-average-checkbox",
            value=False,
            label="Show averages",
            style={
                "display": "block",
                "marginBottom": "0.6rem",
                "fontSize": "1rem",
                "marginLeft": "0.25rem"
            }
        ),

        html.Div([
            html.Label("Select Genre(s)", style={"fontWeight": "bold", "marginTop": "1rem"}),
            dcc.Dropdown(
                id="genre-dropdown",
                options=[],  # Dynamically filled
                multi=True,
                placeholder="Select genres...",
                style={"marginBottom": "1rem"}
            )
        ]),

        html.Div([
            html.Label("Select Year Range", style={"fontWeight": "bold", "marginTop": "1rem"}),
            dcc.RangeSlider(
                id="trends-year-range-slider",
                min=1970,
                max=2024,
                step=1,
                value=[2000, 2020],
                marks={y: str(y) for y in range(1970, 2025, 10)},
                tooltip={"placement": "bottom"}
            ),

            html.Label("Number of Games", style={"fontWeight": "bold", "marginTop": "1.5rem"}),
            dcc.Slider(
                id="trends-num-games-slider",
                min=1000, max=9000, step=500, value=3000,
                marks={i: str(i) for i in range(1000, 9001, 2000)},
                tooltip={"placement": "bottom"}
            )
        ], style={"marginTop": "1rem"})
    ], style={"display": "block" if active_tab == "trends" else "none"})


    # === Final sidebar layout ===

    return html.Div([
        build_search_bar(),
        html.Hr(),
        build_top_control(active_tab),
        html.Hr(),
        html.Div(
            id="middle_options",
            children=[
                games_buttons,
                games_year_slider,
                games_num_slider,
                trends_controls
            ],
            style={"paddingTop": "1rem", "paddingBottom": "1rem"}
        )
    ], style={"padding": "1rem"})



# =========================
# App Layout
# =========================

# Important: Default sidebar must be built immediately at startup!
initial_active_tab = "games"
initial_selected_sub = "most-popular-sub"

app.layout = dbc.Container(
    fluid=True,
    children=[
        # === Shared Stores ===
        dcc.Store(id="active_main_tab", data=initial_active_tab),
        dcc.Store(id="selected_sub_button", data=initial_selected_sub),
        dcc.Store(id="selected_sort_options", data=["Added"]), # start with added, looks best
        dcc.Store(id="selected_genres", data=[]),
        dcc.Store(id="selected_year_range", data=[2000, 2020]),

        # === Trends-specific Stores ===
        dcc.Store(id="selected_game_ids", data=[]),
        dcc.Store(id="trends_sort_metric", data="rating"),
        dcc.Store(id="show_trendline", data=False),
        dcc.Store(id="show_genre_average", data=False),
        dcc.Store(id="trends_year_range", data=[2000, 2020]),
        dcc.Store(id="trends_num_games", data=1000),
        dcc.Store(id="last_clicked_timestamp", data=None),

        dbc.Row([
            # Sidebar
            dbc.Col(
                id="sidebar",
                children=build_sidebar(initial_active_tab, initial_selected_sub, [], []),
                width=3,
                style={
                    "backgroundColor": "#f8f9fa",
                    "height": "100vh",
                    "padding": 0,
                    "borderRight": "1px solid #dee2e6",
                    "display": "flex",
                    "flexDirection": "column"
                }
            ),

            # Data View (main graph + optional comparison panel)
            dbc.Col(build_data_view(), width=9)
        ])
    ]
)

# =========================
# Callbacks
# =========================

@app.callback(
    [Output("active_main_tab", "data"),
     Output("selected_sub_button", "data")],
    [Input("games-button", "n_clicks"),
     Input("trends-button", "n_clicks"),
     Input({"type": "sub-button", "index": ALL}, "n_clicks")],
    [State("active_main_tab", "data"),
     State("selected_sub_button", "data")]
)
def handle_clicks(games_clicks, trends_clicks, sub_clicks, current_tab, selected_sub):
    triggered = ctx.triggered_id

    if triggered == "games-button":
        return "games", "most-popular-sub"
    elif triggered == "trends-button":
        return "trends", dash.no_update
    elif isinstance(triggered, dict) and triggered.get("type") == "sub-button":
        return current_tab, triggered["index"]
    else:
        return current_tab, selected_sub

@app.callback(
    Output("sidebar", "children"),
    [Input("active_main_tab", "data"),
     Input("selected_sub_button", "data"),
     Input("selected_sort_options", "data"),
     Input("selected_genres", "data")]
)
def update_sidebar(active_tab, selected_sub, selected_sort_options, selected_genres):
    return build_sidebar(active_tab, selected_sub, selected_sort_options, selected_genres)

@app.callback(
    Output("selected_genres", "data"),
    Input({"type": "genre-button", "index": ALL}, "n_clicks"),
    State("selected_genres", "data"),
    prevent_initial_call=True
)
def toggle_genre_selection(n_clicks_list, selected_genres):
    triggered = ctx.triggered_id
    if not triggered:
        return dash.no_update

    genre = triggered["index"]
    if genre in selected_genres:
        selected_genres.remove(genre)
    else:
        selected_genres.append(genre)

    return selected_genres

@app.callback(
    Output("selected_year_range", "data"),
    Input("games-year-range-slider", "value"),
    prevent_initial_call=True
)
def update_selected_year_range(year_range):
    return year_range

@app.callback(
    Output("show_trendline", "data"),
    Input("show-trendline-checkbox", "value"),
    prevent_initial_call=True
)
def update_trendline_state(show):
    return show or False

@app.callback(
    Output("show_genre_average", "data"),
    Input("show-genre-average-checkbox", "value")
)
def update_average_checkbox_state(show):
    return show or False

@app.callback(
    Output("trends_year_range", "data"),
    Input("trends-year-range-slider", "value")
)
def update_trends_year_range(value):
    return value

@app.callback(
    Output("trends_num_games", "data"),
    Input("trends-num-games-slider", "value")
)
def update_trends_num_games(value):
    return value

@app.callback(
    Output("genre-dropdown", "options"),
    Input("active_main_tab", "data")
)
def populate_genre_dropdown(tab):
    if tab != "trends":
        return []

    genre_set = set()
    for entry in final_df["genres"].dropna():
        if isinstance(entry, list):
            genre_set.update(entry)
        elif isinstance(entry, str):
            genre_set.update([g.strip() for g in entry.split(",")])
    
    return [{"label": g, "value": g} for g in sorted(genre_set)]

@app.callback(
    [
        Output({"type": "sort-button", "index": ALL}, "value"),
        Output("selected_sort_options", "data")
    ],
    Input({"type": "sort-button", "index": ALL}, "value"),
    State({"type": "sort-button", "index": ALL}, "id"),
    prevent_initial_call=True
)
def enforce_single_games_sort_selection(values, ids):
    # Get which checkbox was most recently checked
    triggered = ctx.triggered_id
    if not triggered:
        # Fall back to first truthy value, or default to 'Added'
        selected = next((id_["index"] for val, id_ in zip(values, ids) if val), "Added")
    else:
        selected = triggered["index"]

    updated_values = [id_["index"] == selected for id_ in ids]
    return updated_values, [selected]


@app.callback(
    [
        Output({"type": "trends-metric", "index": ALL}, "value"),
        Output("trends_sort_metric", "data")
    ],
    Input({"type": "trends-metric", "index": ALL}, "value"),
    State({"type": "trends-metric", "index": ALL}, "id")
)
def enforce_single_metric_selection(values, ids):
    # Find the most recently turned on checkbox (if any)
    triggered = ctx.triggered_id
    if not triggered:
        selected = next((id_["index"] for value, id_ in zip(values, ids) if value), "rating")
        return values, selected

    selected_index = triggered["index"]

    # Update checkboxes so only the selected one is True
    updated_values = [id_["index"] == selected_index for id_ in ids]
    return updated_values, selected_index


# =========================
# MAIN GRAPH – HEAT-MAP VERSION
# =========================

@app.callback(
    Output("main_graph", "children"),
    [
        Input("active_main_tab", "data"),
        Input("selected_sub_button", "data"),
        
        # Heatmap
        Input("selected_sort_options", "data"),
        Input("selected_year_range", "data"),
        Input("num-games-slider", "value"),
        
        # Trends
        Input("trends_year_range", "data"),
        Input("trends_num_games", "data"),
        Input("genre-dropdown", "value"),
        Input("trends_sort_metric", "data"),
        Input("show_trendline", "data"),
        Input("show_genre_average", "data")
    ]
)

def update_main_graph(
    active_tab,
    selected_sub_button,
    selected_sort_options,
    selected_year_range,
    num_games,
    trends_year_range,
    trends_num_games,
    selected_genres,
    trends_sort_metric,
    show_trendline,
    show_genre_average
):
    import squarify
    import statsmodels.api as sm
    import plotly.graph_objects as go
    import pandas as pd
    from dash import html, dcc

    df = final_df.copy()
    df = df.dropna(subset=["name"]).drop_duplicates()
    df["release_year"] = pd.to_datetime(df["released"], errors="coerce").dt.year

    if active_tab == "trends":
        selected_genres = selected_genres or []
        df = df[
            (df["release_year"] >= trends_year_range[0]) &
            (df["release_year"] <= trends_year_range[1])
        ]

        def extract_matching_genre(genre_str):
            try:
                genre_list = [g.strip() for g in genre_str.split(",")]
                for g in genre_list:
                    if g in selected_genres:
                        return g
            except:
                pass
            return None

        df = df.dropna(subset=["released", "rating", "genres"])
        df["released"] = pd.to_datetime(df["released"], errors="coerce")
        df = df[df["rating"] > 0.5]

        if selected_genres:
            df["matched_genre"] = df["genres"].apply(extract_matching_genre)
            df = df[df["matched_genre"].notna()]
            color_column = "matched_genre"
        else:
            df["matched_genre"] = "All"
            color_column = None

        df = df.sample(n=min(trends_num_games, len(df)), random_state=42)
        df = df.sort_values("released")

        y_metric = trends_sort_metric.lower()
        yaxis_label = {
            "rating": "Rating",
            "added": "Added by Users",
            "metacritic": "Metacritic Score",
            "youtube_count": "YouTube Mentions",
            "twitch_count": "Twitch Mentions"
        }.get(y_metric, y_metric.capitalize())

        df[y_metric] = pd.to_numeric(df[y_metric], errors="coerce")
        df = df.dropna(subset=["released", y_metric])

        x = df["released"].map(pd.Timestamp.toordinal)
        y = df[y_metric]

        fig = go.Figure()

        if show_genre_average:
            df["year"] = df["released"].dt.year
            grouped = df.groupby(["year", "matched_genre"])[y_metric].mean().reset_index()
            for genre in grouped["matched_genre"].unique():
                gdf = grouped[grouped["matched_genre"] == genre]
                fig.add_trace(go.Scatter(
                    x=gdf["year"], y=gdf[y_metric],
                    mode="lines+markers", name=genre,
                    marker=dict(size=6, opacity=0.8),
                    line=dict(width=2)
                ))
        else:
            for genre in df["matched_genre"].unique():
                gdf = df[df["matched_genre"] == genre]
                fig.add_trace(go.Scatter(
                    x=gdf["released"], y=gdf[y_metric],
                    mode="markers", name=genre,
                    text=gdf["name"],
                    marker=dict(size=6, opacity=0.6)
                ))

        if show_trendline and not df.empty and len(x) > 1:
            X = sm.add_constant(x)
            model = sm.OLS(y, X).fit()
            trend_y = model.predict(X)
            fig.add_trace(go.Scatter(
                x=df["released"], y=trend_y,
                mode="lines", name="Trendline",
                line=dict(color="orange", width=2)
            ))

        y_min, y_max = y.min(), y.max()
        padding = (y_max - y_min) * 0.05 if y_max > y_min else 1
        yaxis_range = [max(0, y_min - padding), y_max + padding]

        fig.update_layout(
            title=f"Game {yaxis_label} Over Time" + (" by Genre" if selected_genres else ""),
            xaxis_title="Release Date",
            yaxis_title=yaxis_label,
            yaxis_range=yaxis_range,
            transition_duration=500
        )

        return dcc.Graph(figure=fig)

    elif active_tab == "games" and selected_sub_button == "genres-sub":
        column_mapping = {
            "Action", "Adventure", "Arcade", "Board Games", "Card", "Casual", "Educational",
            "Family", "Fighting", "Indie", "Massively Multiplayer", "Platformer", "Puzzle",
            "RPG", "Racing", "Shooter", "Simulation", "Sports", "Strategy"
        }

        if selected_sort_options and selected_sort_options[0] in column_mapping:
            selected_genre = selected_sort_options[0]
        else:
            return html.Div("No genre selected or recognized.", style={"color": "white", "padding": "1rem"})

        df = df[df["release_year"].between(*selected_year_range)]
        df = df[df["genres"].notna()]
        df = df.explode("genres")
        df = df[df["genres"].isin(column_mapping)]
        df = df.rename(columns={"genres": "genre"})
        df = df[df["genre"] == selected_genre]

        sort_by = "rating"
        df = df[df[sort_by].notna() & (df[sort_by] > 0)].sort_values(sort_by, ascending=False).head(50)

        if df.empty:
            return html.Div("No data available for this genre.", style={"color": "white", "padding": "1rem"})

        normed = squarify.normalize_sizes(df[sort_by], 100, 100)
        rects = squarify.squarify(normed, 0, 0, 100, 100)
        df = pd.concat([df.reset_index(drop=True), pd.DataFrame(rects)], axis=1)

        tiles = []
        for _, row in df.iterrows():
            tiles.append(
                html.Div(
                    children=[html.Div([
                        html.Div(row["name"], style={
                            "fontSize": "11px", "fontWeight": "bold",
                            "overflow": "hidden", "textOverflow": "ellipsis", "whiteSpace": "nowrap"
                        }),
                        html.Div(f"{sort_by.capitalize()}: {row[sort_by]:.2f}", style={"fontSize": "10px"})
                    ], style={
                        "position": "absolute", "inset": 0,
                        "backgroundColor": "rgba(0,0,0,0.55)",
                        "display": "flex", "flexDirection": "column",
                        "alignItems": "center", "justifyContent": "center",
                        "padding": "4px"
                    })],
                    style={
                        "position": "absolute",
                        "left": f"{row['x']:.2f}%", "top": f"{row['y']:.2f}%",
                        "width": f"{row['dx']:.2f}%", "height": f"{row['dy']:.2f}%",
                        "backgroundImage": f"url('{row['background_image']}')",
                        "backgroundSize": "cover", "backgroundPosition": "center",
                        "border": "1px solid #fff", "boxSizing": "border-box",
                        "borderRadius": "4px", "overflow": "hidden", "color": "#fff"
                    }
                )
            )

        return html.Div(tiles, style={"position": "relative", "width": "100%", "height": "75vh", "backgroundColor": "#333"})

    elif active_tab == "games" and selected_sub_button == "most-popular-sub":
        df = df[df["release_year"].between(*selected_year_range)]
        column_mapping = {
            "rating": "rating", "youtube": "youtube_count", "twitch": "twitch_count",
            "added": "added", "metacritic": "metacritic"
        }
        sort_by = column_mapping.get(
            (selected_sort_options[0] if selected_sort_options else "rating").lower(),
            "rating"
        )
        df = df.dropna(subset=[sort_by])
        df = df.sort_values(sort_by, ascending=False).head(num_games or 50)

        metric = df[sort_by].astype(float).clip(lower=1e-6)
        normed = squarify.normalize_sizes(metric, 100, 100)
        rects = squarify.squarify(normed, 0, 0, 100, 100)
        df = pd.concat([df.reset_index(drop=True), pd.DataFrame(rects)], axis=1)

        tiles = []
        for _, row in df.iterrows():
            tiles.append(
                html.Div(
                    children=[html.Div([
                        html.Div(row["name"], style={
                            "fontSize": "12px", "fontWeight": "bold",
                            "overflow": "hidden", "textOverflow": "ellipsis", "whiteSpace": "nowrap"
                        }),
                        html.Div(f"{sort_by.capitalize()}: {row[sort_by]:.2f}", style={"fontSize": "10px"})
                    ], style={
                        "position": "absolute", "inset": 0,
                        "backgroundColor": "rgba(0,0,0,0.55)",
                        "display": "flex", "flexDirection": "column",
                        "alignItems": "center", "justifyContent": "center",
                        "padding": "4px"
                    })],
                    style={
                        "position": "absolute",
                        "left": f"{row['x']:.2f}%", "top": f"{row['y']:.2f}%",
                        "width": f"{row['dx']:.2f}%", "height": f"{row['dy']:.2f}%",
                        "backgroundImage": f"url('{row['background_image']}')",
                        "backgroundSize": "cover", "backgroundPosition": "center",
                        "border": "1px solid #fff", "boxSizing": "border-box",
                        "borderRadius": "4px", "overflow": "hidden", "color": "#fff"
                    }
                )
            )

        return html.Div(tiles, style={"position": "relative", "width": "100%", "height": "60vh", "backgroundColor": "#333"})

    else:
        return html.Div("Please select a valid tab or sort option.", style={"padding": "2rem", "color": "gray"})


# =========================
# Run server
# =========================

def open_browser():
    webbrowser.open_new("http://127.0.0.1:8050/")

if __name__ == "__main__":
    threading.Timer(1, open_browser).start()
    app.run(debug=True, use_reloader=False)

